# Goal

Automate first-pass triage using an LLM, extracting structured risk signals from dense, ambiguous legal language.

## Real World Applicaiton

A legal tech startup ingests raw contract clauses from enterprise SaaS agreements. Their intake team manually reviews every clause to flag risk before sending to a lawyer. The goal is to automate that first-pass triage using an LLM.

## Hypothesis

Definition-driven prompting will meaningfully improve accuracy on risk_level and red_flags — fields that require domain reasoning, not just formatting



---



## Install Dependencies

In [1]:
# Dependencies & Setup
# !pip install anthropic pandas

import anthropic, json, pandas as pd
from google.colab import userdata
import os
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
model  = "claude-sonnet-4-6"

## Setup Ground Truth & Benchmark

**Why this matters**

- 3 clauses chosen to stress-test different failure modes
- Liability cap → tests risk_level (model likely underrates it)
- Auto renewal → tests boolean detection buried in boilerplate  
- Indemnification → tests obligation_party hidden in passive voice
- Ground truth = lawyer-level answers

In [11]:
import json
import pandas as pd
import time

# =============================================================
# TEST CLAUSES
# =============================================================

clauses = [
    """
    15. LIMITATION OF LIABILITY
    In no event shall Vendor's total cumulative liability arising out of or related
    to this Agreement exceed the fees paid by Client in the three (3) months
    preceding the claim. This limitation applies regardless of the form of action,
    whether in contract, tort, or otherwise, even if Vendor has been advised of
    the possibility of such damages.
    """,
    """
    8. SUBSCRIPTION RENEWAL
    This Agreement shall automatically renew for successive one (1) year terms
    unless either party provides written notice of non-renewal at least ninety (90)
    days prior to the end of the then-current term. Fees for renewal terms shall
    be subject to increase at Vendor's discretion with thirty (30) days notice.
    """,
    """
    12. INDEMNIFICATION
    Client shall indemnify, defend, and hold harmless Vendor and its officers,
    directors, employees, and agents from and against any and all claims, damages,
    losses, costs, and expenses (including reasonable attorneys fees) arising out
    of or relating to Client's use of the Service, any breach of this Agreement
    by Client, or any third-party claims related to Client data.
    """
]

# =============================================================
# GROUND TRUTH
# =============================================================

ground_truth = [
    {
        "clause_type": "limitation of liability",
        "risk_level": "high",
        "obligation_party": "vendor",
        "auto_renewal": False,
        "red_flags": ["3 month fee cap is very low", "excludes all damage types", "vendor-favoured"]
    },
    {
        "clause_type": "auto renewal",
        "risk_level": "medium",
        "obligation_party": "client",
        "auto_renewal": True,
        "red_flags": ["90 day cancellation window is unusually long", "vendor can raise fees unilaterally"]
    },
    {
        "clause_type": "indemnification",
        "risk_level": "high",
        "obligation_party": "client",
        "auto_renewal": False,
        "red_flags": ["one-sided — client bears all liability", "includes attorneys fees", "covers third-party claims on client data"]
    }
]

# =============================================================
# LLM CALL
# =============================================================

def call_llm(prompt):
    start = time.time()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "output": response.content[0].text,
        "latency": round(time.time() - start, 2),
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

# =============================================================
# PROMPTS
# =============================================================

def build_prompts(text):
    return {
        "naive": f"""
Extract as JSON:
clause_type, risk_level, obligation_party, auto_renewal, red_flags

Text:
{text}
""",
        "schema": f"""
Return only valid JSON matching this schema:
{{
  "clause_type": string,
  "risk_level": string,
  "obligation_party": string,
  "auto_renewal": boolean,
  "red_flags": [string]
}}

Input:
{text}
""",
        "strict": f"""
Return valid JSON only. No explanation, no markdown, no code blocks.

Rules:
- risk_level must be exactly: low, medium, or high
- obligation_party must be exactly: vendor, client, or mutual
- auto_renewal must be a boolean
- red_flags must be a list, empty list if none
- Use null if a value cannot be determined

Input:
{text}
""",
        "definition_few_shot": f"""
Return valid JSON only. No explanation, no markdown, no code blocks.

FIELD DEFINITIONS:
- clause_type: limitation of liability, indemnification, auto renewal,
  termination, data privacy, intellectual property

- risk_level:
  high   = significantly favours vendor, exposes client to uncapped liability
  medium = some risk, negotiable or standard practice
  low    = balanced or client-favourable

- obligation_party:
  vendor = vendor is restricted or liable
  client = client is restricted or liable
  mutual = both parties share obligation

- auto_renewal: true only if contract renews automatically. false otherwise.

- red_flags: specific risks a lawyer should review — uncapped liability,
  one-sided obligations, unilateral fee increases, short cancellation windows.

FEW-SHOT EXAMPLE:
Input:
  "Vendor's liability shall not exceed $500 regardless of the nature
  of the claim. Client waives all rights to consequential damages."

Output:
{{
  "clause_type": "limitation of liability",
  "risk_level": "high",
  "obligation_party": "vendor",
  "auto_renewal": false,
  "red_flags": ["$500 cap is extremely low", "client waives consequential damages"]
}}

Now extract from this input:
{text}
"""
    }

# =============================================================
# SCORER
# =============================================================

def score_extraction(parsed, truth):
    scores = {}

    for field in ["clause_type", "risk_level", "obligation_party"]:
        scores[field] = int(str(parsed.get(field, "")).lower() == str(truth[field]).lower())

    scores["auto_renewal"] = int(parsed.get("auto_renewal") == truth["auto_renewal"])

    raw_predicted = parsed.get("red_flags", [])
    predicted = [str(f).lower() for f in raw_predicted]
    expected  = [str(f).lower() for f in truth["red_flags"]]

    if expected:
        matches = sum(
            any(exp in pred or pred in exp for pred in predicted)
            for exp in expected
        )
        scores["red_flags"] = round(matches / len(expected), 2)
    else:
        scores["red_flags"] = 1.0

    scores["overall"] = round(sum(scores.values()) / len(scores), 2)
    return scores

# =============================================================
# EVALUATION LOOP
# =============================================================

results = []

for i, (text, truth) in enumerate(zip(clauses, ground_truth)):
    for method, prompt in build_prompts(text).items():

        r = call_llm(prompt)

        try:
            cleaned = r["output"].replace("```json", "").replace("```", "").strip()
            parsed = json.loads(cleaned)
            valid_json = True
        except:
            valid_json = False
            parsed = {}

        field_scores = score_extraction(parsed, truth) if valid_json else {
            "clause_type": 0, "risk_level": 0, "obligation_party": 0,
            "auto_renewal": 0, "red_flags": 0, "overall": 0
        }

        results.append({
            "clause": i + 1,
            "method": method,
            "valid_json": valid_json,
            "latency": r["latency"],
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"],
            **field_scores
        })

df = pd.DataFrame(results)
print(df[["clause", "method", "valid_json", "overall"]])

    clause               method  valid_json  overall
0        1                naive        True     0.80
1        1               schema        True     0.80
2        1               strict        True     0.40
3        1  definition_few_shot        True     0.80
4        2                naive        True     0.00
5        2               schema        True     0.20
6        2               strict        True     0.20
7        2  definition_few_shot        True     0.60
8        3                naive        True     0.80
9        3               schema        True     0.87
10       3               strict        True     0.40
11       3  definition_few_shot        True     0.87


**Insight**

- Definition_few_shot won on the hardest clause (auto renewal) but strict underperformed everywhere, more rules ≠ better output. The sweet spot is definitions + examples, not rigid constraints

- CLAUSE 1 naive/schema/definition_few_shot all 0.80, model handles this naturally strict dropped to 0.40. Rigid format rules hurt more than they help

- CLAUSE 2 naive scored 0.00, total failure without guidance definition_few_shot jumped to 0.60. Definitions did the work. This is the hypothesis confirmed, ambiguous clauses need definitions

- CLAUSE 3 schema and definition_few_shot tied at 0.87 strict again dropped to 0.40.

## Results Table


In [19]:
print("FIELD ACCURACY BY METHOD\n")
print(df.groupby("method")[[
    "valid_json", "clause_type", "risk_level",
    "obligation_party", "auto_renewal", "red_flags", "overall"
]].mean().round(2))

print("\n COST & LATENCY BY METHOD\n")
print(df.groupby("method")[["latency", "input_tokens", "output_tokens"]].mean().round(2))

FIELD ACCURACY BY METHOD

                     valid_json  clause_type  risk_level  obligation_party  \
method                                                                       
definition_few_shot         1.0         1.00        0.67              1.00   
naive                       1.0         0.67        0.67              0.67   
schema                      1.0         0.67        0.67              0.67   
strict                      1.0         0.00        0.67              0.67   

                     auto_renewal  red_flags  overall  
method                                                 
definition_few_shot          1.00       0.11     0.76  
naive                        0.67       0.00     0.53  
schema                       1.00       0.11     0.62  
strict                       0.33       0.00     0.33  

 COST & LATENCY BY METHOD

                     latency  input_tokens  output_tokens
method                                                   
definition_few_shot     2

# Analysis Findings


**ACCURACY:**
  
- definition_few_shot top overall at 0.76
  strict worst at 0.33 — constraints alone actively hurt extraction
  clause_type: only definition_few_shot scored 1.00 — definitions
  locked in the correct legal label every time

**BEST FIELD**
  - definition_few_shot 1.00 — knowing vendor/client/mutual explicitly
  solved the passive voice problem completely

**WORST FIELD**
  - all methods scored poorly (0.00-0.11)
  this is a reasoning task not an extraction task
  the model needs legal judgment not just pattern matching
  this is the gap week 4 should target

**COST FINDING:**
  - definition_few_shot uses 2x input tokens vs naive (402 vs 126)
  but is actually faster (2.64s vs 4.43s) and uses far fewer
  output tokens (130 vs 236) — more instruction = more focused output
  better accuracy AND cheaper to run = clear production winner


# Futher Improvements

1. **RED FLAGS IS A REASONING TASK NOT EXTRACTION**
   
  - Scored 0.00-0.11 across all methods.
   The model needs to reason about legal risk not just parse text.
   Fix: chain a second LLM call specifically for red_flags analysis
   after the structured extraction is complete.

2. **EXPAND THE TEST SET**
  - 3 clauses is not enough to draw firm conclusions.
   Edge cases matter — overlapping clause types, mutual obligations,
   and ambiguous renewal language all need coverage.

3. **LLM-AS-JUDGE FOR RED FLAGS**
  - Ground truth labels for red_flags are subjective.
   A lawyer might flag different things than our labels.
   Week 4 should test using Claude to score Claude's red_flags
   against a rubric instead of exact string matching.



---

